In [ ]:
import csv
import pandas as pd
from pathlib import Path
# Find repo root even if notebook is inside /notebooks
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
data = str(ROOT / "data/Energydata export 10-04-2026 12-24-30.csv")
DATA_DIR = ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

In [ ]:
"117_2_P_PVkW | syslab-11/117_2/P_PV | 804158","319_2_P_PVkW | syslab-01/319_2/P_PV | 804159","330_12_P_PV_1kW | syslab-50/330_12/P_PV_1 | 804160","330_12_P_PVBatt_2kW | syslab-50/330_12/P_PVBatt_2 | 804161","330_12_P_PVBatt_3kW | syslab-50/330_12/P_PVBatt_3 | 804162","715_2_P_PVkW | syslab-09/715_2/P_PV | 804163","716_2_P_PVkW | syslab-29/716_2/P_PV | 804164","EGen_GaiaM_ED_P | syslab-50/330_12/P_Gaia | 804165","EGen_AirconM_ED_P | syslab-01/319_2/P_Aircon | 804166"

In [ ]:
def clean_data_nozeros(file_path,resolution,delim=",",save_as_new_file=False):
    """
    reduce time resolution (in minutes!), remove irrelevant columns    """
    # Read the CSV file
    try:
        df = pd.read_csv(file_path,delimiter=delim)
    except Exception as e:
        print(f"Error reading file: {e}")
        return

    df = df.rename(columns={"117_2_P_PVkW | syslab-11/117_2/P_PV | 804158": "B117", 
                            "319_2_P_PVkW | syslab-01/319_2/P_PV | 804159": "B319",
                            "330_12_P_PV_1kW | syslab-50/330_12/P_PV_1 | 804160": "B330_1",
                            "330_12_P_PVBatt_2kW | syslab-50/330_12/P_PVBatt_2 | 804161": "B330_2",
                            "330_12_P_PVBatt_3kW | syslab-50/330_12/P_PVBatt_3 | 804162": "B330_3",
                            "716_2_P_PVkW | syslab-29/716_2/P_PV | 804164": "B716",
                            "715_2_P_PVkW | syslab-09/715_2/P_PV | 804163": "B715"})
    df = df.rename(columns={"EGen_GaiaM_ED_P | syslab-50/330_12/P_Gaia | 804165": "Gaia_WT Power","EGen_AirconM_ED_P | syslab-01/319_2/P_Aircon | 804166":"Aircon_WT Power"})

    #change negative values into 0
    
    df.loc[df["B117"] < 0, "B117"] = 0
    df.loc[df["B319"] < 0, "B319"] = 0
    df.loc[df["B330_1"] < 0, "B330_1"] = 0
    df.loc[df["B330_2"] < 0, "B330_2"] = 0
    df.loc[df["B330_3"] < 0, "B330_3"] = 0
    df.loc[df["B716"] < 0, "B716"] = 0
    df.loc[df["B715"] < 0, "B715"] = 0
    df.loc[df['Aircon_WT Power'] < 0, 'Aircon_WT Power'] = 0
    df.loc[df['Gaia_WT Power'] < 0, 'Gaia_WT Power'] = 0


    df["ts"] = pd.to_datetime(df['ts'])
    df = df.set_index("ts") # set time as index

    df = df.resample(f'{resolution}min').mean() # resize the resolution
    if save_as_new_file==True:
        df.to_csv(f'{file_path[:-4]}_{resolution}min_nozeros.csv', index=True)

    return df



In [ ]:
clean_data_nozeros(data,30,save_as_new_file=True)

In [ ]:
irr_data = ROOT / "data/DMI_radiation.csv" 
ws_data = ROOT / "data/DMI_windspeed.csv" 
price_data = ROOT / "data/DayAheadPrices_04022026_cleaned.csv"

power_data_nozeros = ROOT / "data/Energydata export 10-04-2026 12-24-30_30min_nozeros.csv"
combined_file = ROOT/ "data/combined_data_DMI_gen.csv"

In [ ]:
def combine_data(files:list,filename,save_new_file=True,resolution = False):
    df_combined = pd.read_csv(files[0], 
                     delimiter=',', 
                     decimal='.', 
                     parse_dates=['ts'],  # Your timestamp column name
                     index_col='ts')
    
    for file in files[1:]: 
        df_temp = pd.read_csv(file, 
                     delimiter=',', 
                     decimal='.', 
                     parse_dates=['ts'],  # Your timestamp column name
                     index_col='ts')
        df_combined = df_combined.join(df_temp,how="outer") ## outer = keep ALL timestamps included in ALL the files, inner, left and right still exist! (decide perhaps for inner later on)
        print(f"Added {file}, shape is now: {df_combined.shape}")

    if save_new_file == False: 
        return df_combined

    if resolution != False:
        df_combined = df_combined.resample(f'{resolution}min',).mean() # resize the resolution

    df_combined.to_csv(DATA_DIR / f'{filename}_{resolution}min.csv', index=True, sep =",") ## Save to two folders up and then into data ;) 
    print(f"\nFinal combined shape: {df_combined.shape}")
    print(f"Columns: {list(df_combined.columns)}")



In [ ]:
combine_data([power_data_nozeros,irr_data,ws_data],filename="combined_data_weather_and_power",save_new_file=True,resolution=30)